# Personality Type Prediction – Part 1: EDA and Preprocessing

In this notebook I look at the data first.
Then I build a preprocessing pipeline and test a first model.

In [1]:
import pandas as pd

# load the dataset
df = pd.read_csv("data.csv")

print("Shape:", df.shape)
df.head()

Shape: (19719, 23)


,N6,N8,N7,N1,N9,N10,N3,N5,E3,N2,...,E4,age,C4,E1,A4,E10,E9,gender,hand,target
0,1,1,1,1,1,1,2,1,5,5,...,2,53,1,4,5,1,5,Male,Right,Resilient
1,4,2,3,2,2,4,4,3,3,3,...,3,46,2,2,4,5,1,Female,Right,Moderate
2,5,5,5,5,5,5,5,5,1,1,...,4,14,1,5,5,1,5,Female,Right,Moderate
3,5,5,5,5,4,5,4,4,2,4,...,4,19,5,2,4,5,4,Female,Right,Overcontroller
4,3,3,3,3,3,4,3,3,3,3,...,3,25,3,3,5,5,3,Female,Right,Moderate


## 1. First look at the data

In [2]:
# data types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19719 entries, 0 to 19718
Data columns (total 23 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   N6      19719 non-null  int64
 1   N8      19719 non-null  int64
 2   N7      19719 non-null  int64
 3   N1      19719 non-null  int64
 4   N9      19719 non-null  int64
 5   N10     19719 non-null  int64
 6   N3      19719 non-null  int64
 7   N5      19719 non-null  int64
 8   E3      19719 non-null  int64
 9   N2      19719 non-null  int64
 10  E5      19719 non-null  int64
 11  N4      19719 non-null  int64
 12  E7      19719 non-null  int64
 13  E4      19719 non-null  int64
 14  age     19719 non-null  int64
 15  C4      19719 non-null  int64
 16  E1      19719 non-null  int64
 17  A4      19719 non-null  int64
 18  E10     19719 non-null  int64
 19  E9      19719 non-null  int64
 20  gender  19695 non-null  str  
 21  hand    19619 non-null  str  
 22  target  19719 non-null  str  
dtypes: int64(20), str(3)
m

In [3]:
# how many values are missing per column?
df.isna().sum()

N6          0
N8          0
N7          0
N1          0
N9          0
N10         0
N3          0
N5          0
E3          0
N2          0
E5          0
N4          0
E7          0
E4          0
age         0
C4          0
E1          0
A4          0
E10         0
E9          0
gender     24
hand      100
target      0
dtype: int64

In [4]:
# how many people per personality type?
print(df["target"].value_counts())
print()
print(df["target"].value_counts(normalize=True).round(2))

target
Moderate           8452
Resilient          6163
Overcontroller     2829
Undercontroller    2275
Name: count, dtype: int64

target
Moderate           0.43
Resilient          0.31
Overcontroller     0.14
Undercontroller    0.12
Name: proportion, dtype: float64


## 2. Check the age column

In [5]:
# basic statistics for age
df["age"].describe()

count    1.971900e+04
mean     5.076703e+04
std      7.121272e+06
min      1.300000e+01
25%      1.800000e+01
50%      2.200000e+01
75%      3.100000e+01
max      1.000000e+09
Name: age, dtype: float64

In [6]:
# show all ages above 100, sorted
df.loc[df["age"] > 100, "age"].sort_values().values

array([      118,       188,       188,       191,       208,       211,
             223,       266,      1961,      1964,      1968,      1974,
            1976,      1976,      1977,      1982,      1982,      1982,
            1982,      1984,      1984,      1985,      1985,      1986,
            1986,      1988,      1989,      1989,      1989,      1989,
            1989,      1990,      1990,      1990,      1991,      1991,
            1991,      1992,      1992,      1992,      1992,      1992,
            1992,      1992,      1992,      1992,      1993,      1993,
            1993,      1993,      1993,      1994,      1994,      1994,
            1994,      1994,      1994,      1994,      1994,      1995,
            1995,      1995,      1995,      1995,      1996,      1996,
            1996,      1996,      1996,      1996,      1996,      1997,
            1997,      1997,      1997,      1998,      1998,      1998,
            1998,      1999,      2000,    412434, 

## 3. Clean the data

- Some ages are not realistic (over 100, birth years, very big numbers). I set them to NaN, so the imputer can fill them later.
- The answers should be 1 to 5. A 0 means no answer, so I set it to NaN too.
- I remove duplicate rows.

In [7]:
import numpy as np

# ages over 100 are not realistic -> set to NaN
df.loc[df["age"] > 100, "age"] = np.nan

# all columns with question answers
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

# 0 is not a valid answer -> set to NaN
df[item_cols] = df[item_cols].replace(0, np.nan)

# remove duplicate rows
df = df.drop_duplicates()

print("Shape after cleaning:", df.shape)
df["age"].describe()

Shape after cleaning: (19716, 23)


count    19633.000000
mean        26.264656
std         11.568136
min         13.000000
25%         18.000000
50%         22.000000
75%         31.000000
max        100.000000
Name: age, dtype: float64

## 4. Preprocessing pipeline

The model needs clean numbers without missing values.
- Numbers (question answers and age): fill missing values with the median, then scale.
- Categories (gender and hand): fill missing values with the most common value, then one-hot encode.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# X = features, y = what we want to predict
X = df.drop(columns=["target"])
y = df["target"]

numeric_features = item_cols + ["age"]
categorical_features = ["gender", "hand"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

In [12]:
X_prepared = preprocessor.fit_transform(X)
print("Before:", X.shape, " After:", X_prepared.shape)
# Notice the column count changes — gender/hand became several 0/1 columns.

Before: (19716, 22)  After: (19716, 26)


## 5. First model

I put the preprocessor and a Random Forest together in one pipeline.
I train on 80% of the data and test on the other 20%.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# split into train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# preprocessor + model in one pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.842292089249493


## 6. Try a different model

I change the Random Forest to Logistic Regression and compare the accuracy.

In [14]:
from sklearn.linear_model import LogisticRegression

model_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

model_lr.fit(X_train, y_train)
predictions_lr = model_lr.predict(X_test)
print("Accuracy Logistic Regression:", accuracy_score(y_test, predictions_lr))

Accuracy Logistic Regression: 0.8498985801217038


## 7. Notes

**Why can accuracy be misleading?**
The classes are not balanced. Moderate is 43% of the data. A model that always says "Moderate" already gets 43% accuracy, but it learned nothing. So accuracy alone does not show how good the model is for the small classes.

**What happens if a user does not answer a question?**
The pipeline can handle it. The SimpleImputer fills the missing value with the median.

**Random Forest vs. Logistic Regression**
- Random Forest: 0.842
- Logistic Regression: 0.850

Logistic Regression is a little better. I think this is because the target is made from sums of the answers and thresholds, and Logistic Regression also works with weighted sums.